In [1]:
import numpy as np
import pandas as pd


In [6]:
# Load the datasets
# urldata.csv is our training data (Shreya's baseline)
df = pd.read_csv('urldata.csv')

# majestic_million.csv is our "Whitelist" of safe sites
# We only need the 'Domain' column to save memory
majestic = pd.read_csv('majestic_million.csv', usecols=['Domain'], nrows=100000)

print("Baseline Data Shape:", df.shape)
print("Majestic Whitelist Loaded.")

Baseline Data Shape: (10000, 18)
Majestic Whitelist Loaded.


In [7]:
# Convert the majestic domains into a set for lightning-fast lookups
whitelist = set(majestic['Domain'].unique())

def check_is_popular(domain):
    # If the domain is in our whitelist, it gets a 0 (Safe)
    # If it is NOT in the whitelist, it gets a 1 (Suspicious/Phishing trait)
    if domain in whitelist:
        return 0
    else:
        return 1

# Apply this to our baseline data to update the 'Web_Traffic' feature (Column 11)
# In Shreya's data, this is often column index 10 or 11
df['Web_Traffic'] = df['Domain'].apply(check_is_popular)

print("Updated Web_Traffic feature using Majestic Million.")

Updated Web_Traffic feature using Majestic Million.


In [8]:
import math

def calculate_entropy(text):
    if not text:
        return 0
    text = str(text).lower()
    probs = [float(text.count(c)) / len(text) for c in dict.fromkeys(list(text))]
    entropy = - sum([p * math.log(p) / math.log(2.0) for p in probs])
    # Standard threshold: Phishing URLs usually have entropy > 3.8
    return 1 if entropy > 3.8 else 0

# Create a new column in our dataframe for Entropy
df['Entropy'] = df['Domain'].apply(calculate_entropy)

print("New Feature 'Entropy' added to the dataset.")

New Feature 'Entropy' added to the dataset.


In [11]:
# 1. See all column names
print("Current Columns:", list(df.columns))

# 2. See the first 5 rows of the updated data
df.head()

Current Columns: ['Domain', 'Have_IP', 'Have_At', 'URL_Length', 'URL_Depth', 'Redirection', 'https_Domain', 'TinyURL', 'Prefix/Suffix', 'DNS_Record', 'Web_Traffic', 'Domain_Age', 'Domain_End', 'iFrame', 'Mouse_Over', 'Right_Click', 'Web_Forwards', 'Label', 'Entropy']


,Domain,Have_IP,Have_At,URL_Length,URL_Depth,Redirection,https_Domain,TinyURL,Prefix/Suffix,DNS_Record,Web_Traffic,Domain_Age,Domain_End,iFrame,Mouse_Over,Right_Click,Web_Forwards,Label,Entropy
0,graphicriver.net,0,0,1,1,0,0,0,0,0,0,1,1,0,0,1,0,0,0
1,ecnavi.jp,0,0,1,1,1,0,0,0,0,1,1,1,0,0,1,0,0,0
2,hubpages.com,0,0,1,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0
3,extratorrent.cc,0,0,1,3,0,0,0,0,0,1,0,1,0,0,1,0,0,0
4,icicibank.com,0,0,1,3,0,0,0,0,0,0,0,1,0,0,1,0,0,0


In [12]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Prepare Data: Drop 'Domain' (text) and 'Label' (target) from features
X = df.drop(['Domain', 'Label'], axis=1)
y = df['Label']

# 2. Split into Training (80%) and Testing (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialize XGBoost with Winning Hyperparameters
# We use 1000 trees and depth of 12 to find complex patterns
model = XGBClassifier(
    n_estimators=1000,
    max_depth=12,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss'
)

# 4. Train the model
print("Training the model (this may take a minute)...")
model.fit(X_train, y_train)

# 5. Check Accuracy
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"--- SUCCESS ---")
print(f"Final Model Accuracy: {accuracy * 100:.2f}%")

Training the model (this may take a minute)...
--- SUCCESS ---
Final Model Accuracy: 91.40%


In [13]:
# See which features the AI thinks are most important
importance = pd.DataFrame({'Feature': X.columns, 'Weight': model.feature_importances_})
importance = importance.sort_values(by='Weight', ascending=False)
print(importance)

          Feature    Weight
2      URL_Length  0.679619
9     Web_Traffic  0.118988
1         Have_At  0.066111
16        Entropy  0.036871
0         Have_IP  0.022660
7   Prefix/Suffix  0.016401
15   Web_Forwards  0.008530
3       URL_Depth  0.008280
6         TinyURL  0.007918
11     Domain_End  0.006871
10     Domain_Age  0.006842
12         iFrame  0.006674
13     Mouse_Over  0.005241
4     Redirection  0.004655
8      DNS_Record  0.004338
5    https_Domain  0.000000
14    Right_Click  0.000000


In [21]:
# 1. Remove the dominant 'URL_Length' to find the true weights of the rest
X_inner = X.drop(['URL_Length'], axis=1)
X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(X_inner, y, test_size=0.2, random_state=42)

# 2. Train the Inner Model
inner_model = XGBClassifier(n_estimators=1000, max_depth=10, learning_rate=0.05, eval_metric='logloss')
inner_model.fit(X_train_i, y_train_i)

# 3. Get the new "Inner Weights"
inner_importance = pd.DataFrame({'Feature': X_inner.columns, 'Weight': inner_model.feature_importances_})
print("--- NEW INNER GATE WEIGHTS (The Smart 16) ---")
print(inner_importance.sort_values(by='Weight', ascending=False).head(16))

--- NEW INNER GATE WEIGHTS (The Smart 16) ---
          Feature    Weight
8     Web_Traffic  0.369961
16    Brand_Check  0.172542
1         Have_At  0.093272
0         Have_IP  0.073141
18       TLD_Risk  0.066256
6   Prefix/Suffix  0.037827
15        Entropy  0.037139
17     Word_Count  0.030270
5         TinyURL  0.018717
11         iFrame  0.018378
10     Domain_End  0.015623
9      Domain_Age  0.014477
2       URL_Depth  0.013228
14   Web_Forwards  0.012198
12     Mouse_Over  0.009762
7      DNS_Record  0.008818


In [24]:
import pickle
from google.colab import files

# --- 1. Save and Download the Modernized Dataset ---
# This includes your new columns: Entropy, Brand_Check, etc.
df.to_csv('modernized_phishing_dataset_2024.csv', index=False)
files.download('modernized_phishing_dataset_2024.csv')

# --- 2. Save and Download Model 1 (The Base "Bully" Model) ---
# This is the model with 91.4% accuracy that uses all features
pickle.dump(model, open('phishguard_base_model.pkl', 'wb'))
files.download('phishguard_base_model.pkl')

# --- 3. Save and Download Model 2 (The "Inner Gate" Smart 16) ---
# This is the model that ignores URL_Length to focus on subtle indicators
pickle.dump(inner_model, open('phishguard_inner_model.pkl', 'wb'))
files.download('phishguard_inner_model.pkl')

print("All files are being prepared for download. Check your browser's download folder.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All files are being prepared for download. Check your browser's download folder.
